# Stabolut Delta-Neutral Yield Fund — Quantitative Track Record Audit
**Period:** November 2022 – August 1, 2026 (45 Months)  
**Primary Strategy:** Multi-Exchange Delta-Neutral Perpetual Funding Rate Arbitrage (BitMEX, Binance, Kraken)  
**Tokenized Asset Backed:** USB Token ($1.00 USD Par Value)

This notebook provides an independent, reproducible quantitative audit of the fund's monthly returns, Sharpe Ratio, Sortino Ratio, Drawdown, and Benchmark comparison.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import hashlib

# 1. Load Dataset & Verify SHA-256 Hash
csv_path = '../data/monthly_performance.csv'
with open(csv_path, 'rb') as f:
    sha256 = hashlib.sha256(f.read()).hexdigest()

print(f"Dataset SHA-256 Fingerprint: {sha256}")
df = pd.read_csv(csv_path)
df['Date'] = pd.to_datetime(df['Month'] + '-01')
df.head()

In [ ]:
# 2. Calculate Statistical Performance Metrics
fund_m = df['Yield_Pct'] / 100.0
btc_m = df['BTC_Return_Pct'] / 100.0
sp500_m = df['SP500_Return_Pct'] / 100.0

n_months = len(df)
n_years = n_months / 12.0

# Compounding returns
cum_fund_ret = (df['Fund_NAV'].iloc[-1] / 100.0) - 1.0
cagr_fund = (df['Fund_NAV'].iloc[-1] / 100.0) ** (1.0 / n_years) - 1.0

# Risk-Free Rate = 4.0% p.a.
rf_m = (1.0 + 0.04)**(1/12) - 1.0
fund_vol_ann = fund_m.std(ddof=1) * np.sqrt(12)
sharpe = (fund_m.mean() - rf_m) / fund_m.std(ddof=1) * np.sqrt(12)

print("=== AUDITED PERFORMANCE SUMMARY ===")
print(f"Total Duration: {n_months} Months ({n_years:.2f} Years)")
print(f"Cumulative Net Return: +{cum_fund_ret*100:.2f}%")
print(f"Annualized Return (CAGR): +{cagr_fund*100:.2f}% p.a.")
print(f"Annualized Volatility: {fund_vol_ann*100:.2f}%")
print(f"Sharpe Ratio (Rf=4.0%): {sharpe:.2f}")
print(f"Win Rate (Positive Months): {(fund_m > 0).mean()*100:.1f}%")

In [ ]:
# 3. Plot Cumulative Performance
plt.figure(figsize=(10, 5))
plt.plot(df['Date'], df['Fund_NAV'], label='Stabolut Delta-Neutral Fund', color='#0A58CA', lw=2.5)
plt.plot(df['Date'], df['BTC_NAV'], label='Bitcoin (BTC / USD)', color='#F7931A', lw=1.5, ls='--')
plt.plot(df['Date'], df['SP500_NAV'], label='S&P 500 Index', color='#198754', lw=1.2, ls='-.')
plt.title('Cumulative Compounded NAV (Nov 2022 - Aug 2026)', fontsize=12, fontweight='bold')
plt.ylabel('NAV (USD)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()